# Fine-tune SmolVLA on ALOHA sim — Colab training notebook (Stage B)

Trains `lerobot/smolvla_base` → ALOHA insertion for the **smolvla-edge-nx** project.
Training is pure behavior cloning from a HF dataset — **no MuJoCo / sim needed here**;
evaluation happens back in the repo's Docker env (matched mujoco 2.3.7).

**Runtime → Change runtime type → GPU** (A100 ≈ 4 h at batch 64; T4 works at batch 32–48, slower).
Checkpoints stream to Google Drive so a disconnect never loses more than `SAVE_FREQ` steps.

| stage | where | command |
|---|---|---|
| smoke test (done) | local RTX 2000 Ada | 1k steps, loss 0.34→0.08, checkpoint evals ✔ |
| **full run (this notebook)** | Colab GPU | 20k steps, batch 64 |
| eval + demo GIF | local Docker env | `eval --mode sim` / `make_demo_gif.py` |

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Google Drive — checkpoints persist here across disconnects
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = "/content/drive/MyDrive/smolvla_aloha"   # ~1 GB per checkpoint dir

In [ ]:
# 3. Install — SAME lerobot version as the project's Docker image (0.4.4), so the
#    checkpoint drops straight into `docker compose run --rm eval` back home.
#    Training uses the pyav video backend, so a torch/torchcodec mismatch is harmless.
%pip install -q "lerobot[smolvla]==0.4.4" av
import lerobot, torch
print("lerobot", lerobot.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# 4b. Live loss curves via Weights & Biases (free account: wandb.ai)
# Paste your API key from https://wandb.ai/authorize when prompted.
# To skip wandb entirely, set USE_WANDB = False (loss still lands in Drive via the tee).
USE_WANDB = True
if USE_WANDB:
    import wandb
    wandb.login()
WANDB_FLAGS = "--wandb.enable=true --wandb.project=smolvla-edge" if USE_WANDB else ""


## 5a. Dry run first (~3–5 min) — validate EVERYTHING before spending 4 GPU-hours

200 steps with the identical config. What to check off while it runs:

- [ ] dataset downloads + first batch loads (no rename/backend errors)
- [ ] a **wandb run URL** is printed → open it → `train/loss` chart is updating live
- [ ] console shows `loss:` decreasing (≈0.3–0.4 by step 200, mirroring the local smoke run)
- [ ] a checkpoint appears under `$OUTPUT_DIR-dryrun/checkpoints/000100/`
- [ ] `train_console.log` exists on Drive

All boxes ticked → run the full cell (5b). The dry run writes to a separate `-dryrun`
dir, so the real run starts clean.

In [ ]:
BATCH_SIZE = 64          # A100: 64 | T4 (16 GB): try 48, drop to 32 on OOM
STEPS      = 20000
SAVE_FREQ  = 2000

DRY_DIR = OUTPUT_DIR + "-dryrun"
!lerobot-train \
  --policy.path=lerobot/smolvla_base \
  --policy.push_to_hub=false \
  --dataset.repo_id=lerobot/aloha_sim_insertion_human \
  --dataset.video_backend=pyav \
  --rename_map='{"observation.images.top": "observation.images.camera1"}' \
  --batch_size=$BATCH_SIZE \
  --steps=200 \
  --save_freq=100 \
  $WANDB_FLAGS \
  --output_dir=$DRY_DIR 2>&1 | tee -a $DRY_DIR-console.log

!ls $DRY_DIR/checkpoints/ && echo "DRY RUN OK — checkpoint saved"

In [ ]:
# 5b. FULL RUN (~4 h on A100). Same config as the dry run, full step budget.
# 4. (optional) HF login for faster downloads / higher rate limits — not required
# from huggingface_hub import notebook_login; notebook_login()

# 5. Train. Mirrors scripts/train.sh exactly (see repo) — key flags:
#    --rename_map: smolvla_base declares camera1..3; the ALOHA dataset has a single top camera
#    --dataset.video_backend=pyav: avoids the torchcodec/torch ABI coupling
#    --policy.push_to_hub=false: keep the checkpoint local/Drive
!lerobot-train \
  --policy.path=lerobot/smolvla_base \
  --policy.push_to_hub=false \
  --dataset.repo_id=lerobot/aloha_sim_insertion_human \
  --dataset.video_backend=pyav \
  --rename_map='{"observation.images.top": "observation.images.camera1"}' \
  --batch_size=$BATCH_SIZE \
  --steps=$STEPS \
  --save_freq=$SAVE_FREQ \
  $WANDB_FLAGS \
  --output_dir=$OUTPUT_DIR 2>&1 | tee -a $OUTPUT_DIR/train_console.log

# The loss curve is NOT saved by lerobot (checkpoints hold weights + resume state only);
# the tee above keeps the console log — with all `loss:` lines — on Drive. Plot it later:
#   grep -oE "step:[0-9]+ .*loss:[0-9.]+" train_console.log

In [ ]:
# 6. If the runtime disconnected mid-run: reconnect, re-run cells 1-3, then resume
#    from the last saved checkpoint instead of re-running cell 5:
# !lerobot-train \
#   --config_path=$OUTPUT_DIR/checkpoints/last/pretrained_model/train_config.json \
#   --resume=true

In [ ]:
# 7. Package the final checkpoint for the trip home (skip if you'll pull from Drive directly)
!ls -la $OUTPUT_DIR/checkpoints/
!cd $OUTPUT_DIR/checkpoints && zip -qr /content/smolvla_aloha_last.zip last/ && ls -la /content/smolvla_aloha_last.zip
# then: Files panel -> download, or copy the zip into Drive

## Back on the dev box

```bash
# unzip into the repo
unzip smolvla_aloha_last.zip -d outputs/train/smolvla_aloha/checkpoints/

# the success-rate deliverable (task 2.5):
docker compose run --rm shell python -m smolvla_edge.eval --mode sim \
    --policy-path outputs/train/smolvla_aloha/checkpoints/last \
    --env-id gym_aloha/AlohaInsertion-v0 --episodes 20 \
    --task "insert the peg into the socket"

# regenerate the demo GIF with YOUR policy:
docker compose run --rm shell python scripts/make_demo_gif.py --mode rollout \
    --policy-path outputs/train/smolvla_aloha/checkpoints/last \
    --env-id gym_aloha/AlohaInsertion-v0 --task "insert the peg into the socket"
```

Checkpoint facts (identical regardless of the training GPU): **450 M params, ~865 MB**
(bf16-stored). The eval pipeline auto-uses the checkpoint's saved processors
(rename / tokenize / normalize), verified end-to-end with the local smoke checkpoint.